In [ ]:
curr_dir ='/content/drive/MyDrive/BigData/CIE427_Mini-Project2_TeamX'

In [ ]:
#@title SETUP { display-mode: "form" }
!wget -q http://archive.apache.org/dist/spark/spark-2.3.1/spark-2.3.1-bin-hadoop2.7.tgz
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!tar xf spark-2.3.1-bin-hadoop2.7.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-2.3.1-bin-hadoop2.7"
import findspark
findspark.init()

import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate() 
spark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran40/ InRelease [3,626 B]
Ign:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  InRelease
Ign:3 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  Release [696 B]
Hit:5 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  Release
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  Release.gpg [836 B]
Get:7 http://security.ubuntu.com/ubuntu bionic-security InRelease [88.7 kB]
Get:8 http://ppa.launchpad.net/c2d4u.team/c2d4u4.0+/ubuntu bionic InRelease [15.9 kB]
Hit:9 http://archive.ubuntu.com/ubuntu bionic InRelease
Get:10 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran40/ Packages [73.9 kB]
Get:11 http://archive.ubuntu.com/ubuntu bionic-updates InRelease [88.7 kB]
Hit:13 http://ppa.launchpad.net/cran/

In [ ]:
import requests
import requests
import json
import time
import pandas as pd
import random
import time
from ast import literal_eval as eval
import itertools


##Get Champion with Class

In [ ]:

api = "RGAPI-f38e3756-49f8-40f2-b9b9-d34c454a628c"
region = "americas"


def get__list_by_puuid(puuid,region,api):
    """  get a list of solo/due ranked matches given player puuid """
    url = f'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?type=ranked&count=25&queue=420&api_key={api}'
    match_list = requests.get(url).json()
    time.sleep(0.8)
    return match_list

def get_match_by_match_id(match_id,region,api):
    """  get match info match id """
    url= f'https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}?api_key={api}'
    match = requests.get(url).json()
    time.sleep(0.8)
    return match

def rem_item(lst,item):
    """ remove item from list if exists """
    if type(item)!=list:
        item = [item]
    for i in item:
        try:
            lst.remove(i)
        except ValueError:
            pass


In [ ]:
url = 'http://ddragon.leagueoflegends.com/cdn/9.3.1/data/en_US/champion.json'
json_list = requests.get(url).json()

In [ ]:
all_champs = json_list['data']


In [ ]:
all_champs['Aatrox'].keys()

dict_keys(['version', 'id', 'key', 'name', 'title', 'blurb', 'info', 'image', 'tags', 'partype', 'stats'])

In [ ]:
# get the match objects for the seed match ids
dataset=[]
for match in match_ids:
    dataset.append(get_match_by_match_id(match,region,api))



In [ ]:
# get the participants of these matches and remove duplicates 
import itertools
new_puuids = list(itertools.chain.from_iterable([dataset[i]['metadata']['participants'] for i in range(len(dataset))]))
new_puuids= list(set(new_puuids))
rem_item(new_puuids, seed_puuids)
len(new_puuids)


In [ ]:
# loop get the matches of the participants
#matches=[]
for p in new_puuids:
    match_ids.extend(get_match_list_by_puuid(p, region,api) )

In [ ]:
# remove duplicate matches
match_ids = list(set(match_ids))

In [ ]:
# get the match objects for the remaining match ids to complete 75000 <THIS TAKES A WHILE>
for match in match_ids[300:75000]:
    dataset.append(get_match_by_match_id(match, region, api))

In [ ]:
# save dataset to csv
df1 = pd.DataFrame.from_dict(dataset)
info = list(df1['info'])
metadata= list(df1['metadata'])

In [ ]:
info_df = pd.DataFrame(info)
info_df.head()

,gameCreation,gameDuration,gameEndTimestamp,gameId,gameMode,gameName,gameStartTimestamp,gameType,gameVersion,mapId,participants,platformId,queueId,teams,tournamentCode
0,1637007192000,3052,1.637010e+12,1167194809,CLASSIC,teambuilder-match-1167194809,1637007286384,MATCHED_GAME,11.22.406.3587,11,"[{'assists': 14, 'baronKills': 0, 'bountyLevel...",LA1,420,"[{'bans': [{'championId': 11, 'pickTurn': 1}, ...",
1,1637003238000,1765,1.637005e+12,1167128147,CLASSIC,teambuilder-match-1167128147,1637003368554,MATCHED_GAME,11.22.406.3587,11,"[{'assists': 10, 'baronKills': 0, 'bountyLevel...",LA1,420,"[{'bans': [{'championId': 51, 'pickTurn': 1}, ...",
2,1636854478000,1690,1.636856e+12,1166298822,CLASSIC,teambuilder-match-1166298822,1636854637451,MATCHED_GAME,11.22.406.3587,11,"[{'assists': 3, 'baronKills': 0, 'bountyLevel'...",LA1,420,"[{'bans': [{'championId': 25, 'pickTurn': 1}, ...",
3,1636840811000,2194,1.636843e+12,1166206602,CLASSIC,teambuilder-match-1166206602,1636840903310,MATCHED_GAME,11.22.406.3587,11,"[{'assists': 11, 'baronKills': 0, 'bountyLevel...",LA1,420,"[{'bans': [{'championId': 360, 'pickTurn': 1},...",
4,1636838808000,1470,1.636840e+12,1166212749,CLASSIC,teambuilder-match-1166212749,1636838873138,MATCHED_GAME,11.22.406.3587,11,"[{'assists': 6, 'baronKills': 0, 'bountyLevel'...",LA1,420,"[{'bans': [{'championId': 64, 'pickTurn': 1}, ...",


In [ ]:
meta_df = pd.DataFrame(metadata)
meta_df.head()

,dataVersion,matchId,participants
0,2,LA1_1167194809,[AhJHcTU-faewtWeEkt94GxZxpKDRhv2zyY3LLmNnIcAI3...
1,2,LA1_1167128147,[2UdVXj_zxs-7kJoehZl6IaDduQgl_Rq2kqj5q2Zxfub7e...
2,2,LA1_1166298822,[l2zZlpaPydHr7T81pmVdZLlVuRyCuayb-2-tvo-kaQVaR...
3,2,LA1_1166206602,[-UmCsuRmeH51BBnKXIgj8qOLItWc8NaW0cSkzOQDnhFKE...
4,2,LA1_1166212749,[oVS6v55WjK8ZFNRtLiH1ZP9SkziipbUpwCs907w1cLzgp...


In [ ]:
info_df.to_csv('match_info.csv')
meta_df.to_csv('match_meta.csv')


## Reformat data

In [ ]:
info_df = spark.read.option('header', 'true').csv('match_info.csv')
info_df.show()

+---+-------------+------------+----------------+----------+--------+--------------------+------------------+------------+--------------+-----+--------------------+----------+-------+--------------------+--------------+
|_c0| gameCreation|gameDuration|gameEndTimestamp|    gameId|gameMode|            gameName|gameStartTimestamp|    gameType|   gameVersion|mapId|        participants|platformId|queueId|               teams|tournamentCode|
+---+-------------+------------+----------------+----------+--------+--------------------+------------------+------------+--------------+-----+--------------------+----------+-------+--------------------+--------------+
|  0|1637007192000|        3052| 1637010338696.0|1167194809| CLASSIC|teambuilder-match...|     1637007286384|MATCHED_GAME|11.22.406.3587|   11|[{'assists': 14, ...|       LA1|    420|[{'bans': [{'cham...|          null|
|  1|1637003238000|        1765| 1637005134107.0|1167128147| CLASSIC|teambuilder-match...|     1637003368554|MATCHED_GAM

In [ ]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType

participants_df = info_df[['participants']]
participants_df.show()

+--------------------+
|        participants|
+--------------------+
|[{'assists': 14, ...|
|[{'assists': 10, ...|
|[{'assists': 3, '...|
|[{'assists': 11, ...|
|[{'assists': 6, '...|
|[{'assists': 2, '...|
|[{'assists': 0, '...|
|[{'assists': 13, ...|
|[{'assists': 4, '...|
|[{'assists': 1, '...|
|[{'assists': 0, '...|
|[{'assists': 10, ...|
|[{'assists': 6, '...|
|[{'assists': 0, '...|
|[{'assists': 3, '...|
|[{'assists': 5, '...|
|[{'assists': 3, '...|
|[{'assists': 1, '...|
|[{'assists': 1, '...|
|[{'assists': 5, '...|
+--------------------+
only showing top 20 rows



In [ ]:
all_participants = list(info_df['participants'])
len(all_participants)

In [ ]:
all_participants = itertools.chain.from_iterable(all_participants)


In [ ]:
all_participants =list(all_participants)

In [ ]:
participants = pd.DataFrame(all_participants)
participants.head()